In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi_loc = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018
doi_lma = os.path.join(raw, '10.15485.1618132') # Leaf mass per area and leaf water content measurements from field survey in association with NEON AOP survey, East River, CO 2018
doi_chem = os.path.join(raw, '10.15485.1631278') # Site-level Foliar C, N, delta13C data from samples collected during field survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'leaf_properties'

In [3]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
29,leaf_properties,trait,USER-DEFINED
30,leaf_properties,value,double precision
31,leaf_properties,error_precision,character
32,leaf_properties,method,USER-DEFINED
33,leaf_properties,handling,USER-DEFINED
34,leaf_properties,units,USER-DEFINED
35,leaf_properties,notes,character
36,leaf_properties,sample_id,uuid
37,leaf_properties,trait_id,uuid


In [4]:
# view relevant data-types info
# for now we can't do this automatically by column_name, because column_name is not explicitly linked to enum_type in any way yet

data_types = data_types[data_types['Enum Type'].isin(['Trait', 'Trait_method', 'Sample_handling', 'Trait_units'])]
data_types

,Schema,Enum Type,Enum Value
25,sbgplants,Sample_handling,Flash frozen
26,sbgplants,Sample_handling,Fresh
27,sbgplants,Sample_handling,Oven dried
28,sbgplants,Trait,Aluminum
29,sbgplants,Trait,Boron
30,sbgplants,Trait,CRF
31,sbgplants,Trait,Calcium
32,sbgplants,Trait,Chl
33,sbgplants,Trait,Copper
34,sbgplants,Trait,Iron


In [61]:
# load relevant plot/sample/spp data

# get species_or_type from spp list
species_list = pd.read_csv(os.path.join(doi_loc, 'species_list.csv'))
species_list['species_or_type'] = species_list['Genus'] + ' ' + species_list['Species']
species_list.loc[species_list['species_or_type'].isna(), 'species_or_type'] = species_list.loc[species_list['Genus'].isna(), 'CoverCode']
species_list = species_list[['CoverCode', 'species_or_type']]

# prep sample_list to get sample_id
sample_list = pd.read_csv(os.path.join(out_folder, 'sample_list.csv'))[['plot_name', 'species_or_type', 'sample_id']]

# prep sample_site to get sample_id for meadows
sample_site = pd.read_csv(os.path.join(doi_loc, 'sample_site.csv'))[['SamplingArea', 'SampleSiteCode']]

# prep sampling_area to get sample_id
# sampling_area = pd.read_csv(os.path.join(doi_loc, 'sampling_area.csv'))

In [62]:
# load relevant trait data - process each of the 3 trait datasets separately and then concatenate

# start with the easiest set - trees. One sample per entry in sample_list
lma_site = pd.read_csv(os.path.join(doi_lma, 'lma_site_samples.csv'))
# fix typo
lma_site.loc[lma_site.Species=='engelmann', 'Species'] = 'Engelmann'
# join species_or_type
lma_site = pd.merge(lma_site, species_list, left_on='Species', right_on='CoverCode', how='left', suffixes=('',''))

# assign sample_id
lma_site = pd.merge(lma_site, sample_list, left_on=['SampleSiteCode','species_or_type'], right_on=['plot_name','species_or_type'], how='left', suffixes=('',''))

# short to long format
lma_site = lma_site[['Wet_Weight_g', 'Dry_Weight_g', 'LMA_gm2', 'LWC_%', 'sample_id']]
lma_site = pd.melt(lma_site, id_vars=['sample_id'], value_vars=['Wet_Weight_g', 'Dry_Weight_g', 'LMA_gm2', 'LWC_%'], var_name='trait', value_name='value')

# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['trait'] = lma_site['trait']
out_table['value'] = lma_site['value']
out_table['sample_id'] = lma_site['sample_id']

out_table['method'] = 'Destructive' # double-check

# map trait names
traits = {
    'Wet_Weight_g': 'wet weight',
    'Dry_Weight_g': 'dry weight',
    'LMA_gm2': 'LMA',
    'LWC_%': 'LWC'
}
out_table['trait'] = out_table['trait'].map(traits)

# map sample handling
handling = {
    'wet weight': 'Fresh',
    'dry weight': 'Oven Dried',
    'LMA': 'Oven Dried', # ?
    'LWC': pd.NA # ?
}
out_table['handling'] = out_table['trait'].map(handling)

# map units
units = {
    'wet weight': 'g',
    'dry weight': 'g',
    'LMA': 'grams dry mass per g m2',
    'LWC': 'percentage'
}
out_table['units'] = out_table['trait'].map(units)

# filter na rows
out_table = out_table[out_table.value.isna()==False]

out_table_lmasite = out_table.copy()

In [67]:
# repeat for meadow traits
# this is the tricky one - rows in leaf_properties will be duplicated for multiple sample_ids?
# this is weird and maybe pseudo-replication?
# hold off on this until talk to Dana

lma_meadow = pd.read_csv(os.path.join(doi_lma, 'lma_meadow_area_samples.csv'))

# join species_or_type to lma_meadow
lma_meadow = pd.merge(lma_meadow, species_list, left_on='SpeciesCode', right_on='CoverCode', how='left', suffixes=('',''))

# join SamplingArea to sample_list
sample_list_ = pd.merge(sample_list, sample_site, left_on=['plot_name'], right_on=['SampleSiteCode'], how='left', suffixes=('',''))

# get all plots per SampleArea/species_or_type
sampleid_key = (
    sample_list_
    .groupby(['SamplingArea', 'species_or_type'])['sample_id']
    .agg(list)
    .reset_index(name='sample_id')
)
# assign sample ids
lma_meadow = lma_meadow.merge(
    sampleid_key,
    left_on=['SampleArea', 'species_or_type'],
    right_on=['SamplingArea', 'species_or_type'],
    how='left'
)

# short to long format
lma_meadow = lma_meadow[['Wet Weight (g)', 'Dry Weight (g)', 'LMA (g/m2)', 'LWC (%)', 'sample_id']]
lma_meadow = pd.melt(lma_meadow, id_vars=['sample_id'], value_vars=['Wet Weight (g)', 'Dry Weight (g)', 'LMA (g/m2)', 'LWC (%)'], var_name='trait', value_name='value')

# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['trait'] = lma_meadow['trait']
out_table['value'] = lma_meadow['value']
out_table['sample_id'] = lma_meadow['sample_id']

out_table['method'] = 'Destructive' # double-check

# map trait names
traits = {
    'Wet Weight (g)': 'wet weight',
    'Dry Weight (g)': 'dry weight',
    'LMA (g/m2)': 'LMA',
    'LWC (%)': 'LWC'
}
out_table['trait'] = out_table['trait'].map(traits)

# map sample handling
handling = {
    'wet weight': 'Fresh',
    'dry weight': 'Oven Dried',
    'LMA': 'Oven Dried', # ?
    'LWC': pd.NA # ?
}
out_table['handling'] = out_table['trait'].map(handling)

# map units
units = {
    'wet weight': 'g',
    'dry weight': 'g',
    'LMA': 'grams dry mass per g m2',
    'LWC': 'percentage'
}
out_table['units'] = out_table['trait'].map(units)

# filter na rows
out_table = out_table[out_table.value.isna()==False]

out_table_lmameadow = out_table.copy()

In [88]:
# repeat for foliar chemistry
chem = pd.read_csv(os.path.join(doi_chem, 'CN_Results_Foliar.csv'))

# get all sample_ids per plot
sampleid_key = (
    sample_list_
      .groupby('plot_name', sort=False)['sample_id']
      .agg(list)              # unique, order-preserving
)
# assign sample ids
chem['sample_id'] = chem['SampleSiteCode'].map(sampleid_key)

# short to long format
chem = chem[['N_weight_percent', 'sample_id']] # no C_weight_percent or d13C?
chem = pd.melt(chem, id_vars=['sample_id'], value_vars=['N_weight_percent'], var_name='trait', value_name='value')

# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['trait'] = chem['trait']
out_table['value'] = chem['value']
out_table['sample_id'] = chem['sample_id']

out_table['method'] = 'Destructive' # double-check

# map trait names
traits = {
    'N_weight_percent': 'Nitrogen'
}
out_table['trait'] = out_table['trait'].map(traits)

# map sample handling
handling = {
    'Nitrogen': 'Flash frozen'
}
out_table['handling'] = out_table['trait'].map(handling)

# map units
units = {
    'Nitrogen': 'concentration in percent dry mass' # vs just percent?
}
out_table['units'] = out_table['trait'].map(units)

# # filter na rows
# out_table = out_table[out_table.value.isna()==False]

out_table_chem = out_table.copy()

out_table_chem

,trait,value,error_precision,method,handling,units,notes,sample_id,trait_id
0,Nitrogen,3.35,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,"[sample_1, sample_2, sample_3]",NaN
1,Nitrogen,4.53,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,"[sample_4, sample_5, sample_6, sample_7, sampl...",NaN
2,Nitrogen,3.28,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,"[sample_9, sample_10, sample_11, sample_12]",NaN
3,Nitrogen,4.02,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,"[sample_13, sample_14, sample_15, sample_16, s...",NaN
4,Nitrogen,4.35,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,"[sample_18, sample_19, sample_20]",NaN
...,...,...,...,...,...,...,...,...,...
468,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1257],NaN
469,Nitrogen,1.26,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1258],NaN
470,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1259],NaN
471,Nitrogen,0.86,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1260],NaN


In [89]:
# join the three datasets

out_table = pd.concat([out_table_lmasite, out_table_lmameadow, out_table_chem])
out_table

,trait,value,error_precision,method,handling,units,notes,sample_id,trait_id
0,wet weight,2.08,NaN,Destructive,Fresh,g,NaN,sample_83,NaN
1,wet weight,1.34,NaN,Destructive,Fresh,g,NaN,sample_84,NaN
2,wet weight,1.65,NaN,Destructive,Fresh,g,NaN,sample_85,NaN
3,wet weight,1.62,NaN,Destructive,Fresh,g,NaN,sample_86,NaN
4,wet weight,1.70,NaN,Destructive,Fresh,g,NaN,sample_87,NaN
...,...,...,...,...,...,...,...,...,...
468,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1257],NaN
469,Nitrogen,1.26,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1258],NaN
470,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1259],NaN
471,Nitrogen,0.86,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1260],NaN


In [90]:
# confirm final column data types

print(out_table.dtypes)

# adjust as necessary

out_table.dtypes

trait               object
value              float64
error_precision     object
method              object
handling            object
units               object
notes               object
sample_id           object
trait_id            object
dtype: object


trait               object
value              float64
error_precision     object
method              object
handling            object
units               object
notes               object
sample_id           object
trait_id            object
dtype: object

In [91]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)

In [92]:
out_table

,trait,value,error_precision,method,handling,units,notes,sample_id,trait_id
0,wet weight,2.08,NaN,Destructive,Fresh,g,NaN,sample_83,NaN
1,wet weight,1.34,NaN,Destructive,Fresh,g,NaN,sample_84,NaN
2,wet weight,1.65,NaN,Destructive,Fresh,g,NaN,sample_85,NaN
3,wet weight,1.62,NaN,Destructive,Fresh,g,NaN,sample_86,NaN
4,wet weight,1.70,NaN,Destructive,Fresh,g,NaN,sample_87,NaN
...,...,...,...,...,...,...,...,...,...
468,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1257],NaN
469,Nitrogen,1.26,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1258],NaN
470,Nitrogen,1.30,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1259],NaN
471,Nitrogen,0.86,NaN,Destructive,Flash frozen,concentration in percent dry mass,NaN,[sample_1260],NaN
